# Building Age Classification from Satellite Imagery

# Notebook 4 — Modelling & Evaluation

---

**Approach: Metric Learning**

In this notebook we will use a **metric learning** approach to the classification problem. This involves learning a feature representation where
buildings of the same age class cluster together, regardless of which city they come from. 

In practice this means:
1. **Train on Madrid** (the labelled city) using cross-validation to get reliable performance
   estimates with error bars.
2. **Zero-shot transfer** — apply the Madrid model directly to Amsterdam, with no Amsterdam
   labels, to measure the baseline domain gap.
3. **Few-shot prototype transfer** — given a small number of labelled Amsterdam examples,
   compute per-class prototype vectors (class means in feature space) and classify by
   nearest prototype. This is the inference step of prototype networks, a standard
   metric learning technique.

The few-shot experiment shows how much label budget is needed to close the Madrid–Amsterdam gap.

For more information on the metric learning approach, please see the supportind document "METRIC_LEARNING_APPROACH.md"

**NOTE**:  Metric learning is _one possible approach_.  Feel free to try another approach!

## 4.1 Setup & Imports

In [1]:
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
from pathlib import Path

import tensorflow as tf
from tensorflow.keras import layers, models, losses, optimizers, callbacks
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)

CLASS_COLORS = {1: '#4393c3', 2: '#2ca25f', 3: '#fd8d3c', 4: '#de2d26'}
CLASS_LABELS_MADRID    = {1: 'Pre-1960',  2: '1960-1984', 3: '1984-2004', 4: '2004-2024'}
CLASS_LABELS_AMSTERDAM = {1: 'Pre-1945',  2: '1945-1984', 3: '1984-2004', 4: '2004-2024'}

plt.rcParams['figure.dpi'] = 100
print("Setup complete.")


I0000 00:00:1792961650.663883  448521 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1792961651.052572  448521 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792961652.929554  448521 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Setup complete.


## 4.2 Load Preprocessed Data

In [2]:
with open('data/preprocessed/preprocessed_data.pkl', 'rb') as f:
    data = pickle.load(f)

X_madrid    = data['X_madrid']
y_madrid    = data['y_madrid']
X_amsterdam = data['X_amsterdam']
y_amsterdam = data['y_amsterdam']
feature_names = data['feature_names']
pixel_ids_madrid    = data['pixel_ids_madrid']     # (N, 2)  px_key, py_key
pixel_ids_amsterdam = data['pixel_ids_amsterdam']  # (N, 2)  px_key, py_key

print(f'Madrid:    {X_madrid.shape}  ({len(np.unique(y_madrid))} classes)')
print(f'Amsterdam: {X_amsterdam.shape}  ({len(np.unique(y_amsterdam))} classes)')
print(f'Features:  {len(feature_names)}')
print()
for name, y, labels in [('Madrid', y_madrid, CLASS_LABELS_MADRID),
                         ('Amsterdam', y_amsterdam, CLASS_LABELS_AMSTERDAM)]:
    print(f'{name} class distribution:')
    for cls in sorted(np.unique(y)):
        n = (y == cls).sum()
        print(f'  Class {cls} ({labels[cls]}): {n:,}  ({n/len(y)*100:.1f}%)')
    print()
y_madrid = y_madrid - 1

Madrid:    (76263, 60)  (4 classes)
Amsterdam: (25992, 60)  (4 classes)
Features:  60

Madrid class distribution:
  Class 1 (Pre-1960): 13,794  (18.1%)
  Class 2 (1960-1984): 27,152  (35.6%)
  Class 3 (1984-2004): 18,672  (24.5%)
  Class 4 (2004-2024): 16,645  (21.8%)

Amsterdam class distribution:
  Class 1 (Pre-1945): 7,568  (29.1%)
  Class 2 (1945-1984): 8,802  (33.9%)
  Class 3 (1984-2004): 6,745  (26.0%)
  Class 4 (2004-2024): 2,877  (11.1%)



In [3]:
NUM_CLASSES = len(np.unique(y_madrid))

In [5]:
# create a function that returns a model instance 
def create_model():
    model = models.Sequential([
        layers.Input(shape=(X_madrid.shape[1],)),
        
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(rate=0.3),

        layers.Dense(64, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(rate=0.2),
        
        layers.Dense(32, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.2),

        layers.Dense(NUM_CLASSES, activation='softmax')
    ])

    model.compile(
        optimizer = optimizers.Adam(learning_rate=0.005),
        # using sparse cross entropy since y contains interger labels 
        loss = losses.SparseCategoricalCrossentropy(),
        metrics=['accuracy']
    )

    return model

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

fold_accuracies = []
fold_fi_scores = []

# continueross validation loop 
for fold, (train_ids, val_ids) in enumerate(cv.split(X_madrid, y_madrid)):
    print(f"\n--- Fold {fold + 1} ---")

    # Split arrays by current indices 
    X_train, X_val = X_madrid[train_ids], X_madrid[val_ids]
    y_train, y_val = y_madrid[train_ids], y_madrid[val_ids]

    # Instantiate a clean architecture and train 
    model = create_model()

    # Define EarlyStopping 
    early_stopping = callbacks.EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True
    )
    history = model.fit(
        X_train, y_train, 
        validation_data =(X_val, y_val),
        epochs=100,
        batch_size=64,
        callbacks=[early_stopping],
        verbose=0
    )

    # Standard evaluation (loss and accuracy)
    scores = model.evaluate(X_val, y_val, verbose=0)
    val_acc = scores[1]

    # Macro_f1 evaluation step 
    # get probability arrays from the network 
    y_pred_probs = model.predict(X_val, verbose=0)
    # convert probabilities to distinct classes 
    y_pred = np.argmax(y_pred_probs, axis=1)

    # Calculate macro_f1 comparing the correct labels to predictions 
    val_f1 = f1_score(y_val, y_pred, average="macro")

    print(f"Fold {fold + 1} - Accuracy: {val_acc:.4f} | Macro_F1: {val_f1:.4f}")

    fold_accuracies.append(val_acc)
    fold_fi_scores.append(val_f1)

    # Clean up graph state to prevent slowdowns 
    tf.keras.backend.clear_session()

# Aggregated performance across all folds 
print("\n--- FINAL RESULTS ---")
print(f"Average CV Accuracy: {np.mean(fold_accuracies):.4f} +/- {np.std(fold_accuracies)}")
print(f"Average Macro F1: {np.mean(fold_fi_scores):.4f} +/- {np.std(fold_fi_scores):.4f}")


--- Fold 1 ---


E0000 00:00:1792961916.100894  448521 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Fold 1 - Accuracy: 0.6445 | Macro_F1: 0.6259

--- Fold 2 ---
Fold 2 - Accuracy: 0.6550 | Macro_F1: 0.6424

--- Fold 3 ---
Fold 3 - Accuracy: 0.6531 | Macro_F1: 0.6377

--- Fold 4 ---
Fold 4 - Accuracy: 0.6465 | Macro_F1: 0.6318

--- Fold 5 ---
Fold 5 - Accuracy: 0.6399 | Macro_F1: 0.6268

--- FINAL RESULTS ---
Average CV Accuracy: 0.6478 +/- 0.005546110810721935
Average Macro F1: 0.6329 +/- 0.0063
